# 04 — Merge BOM + Trend Drivers\n\nMap bottom-up (BOM) and top-down (Trend) driver lists onto each other.\n- **Match** (both sources) → confidence: high, tag: validated\n- **BOM-only** → confidence: medium, tag: incremental\n- **Trend-only** → confidence: low, tag: speculative/disruptive

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from src.llm import safe_chat_json
from src.traceability import TechDriver, DriverOrigin, DriverConfidence, stable_id
from src import prompts

# load BOM drivers
with open("../data/outputs/bom_state.json") as f:
    bom_state = json.load(f)
bom_drivers = [TechDriver(**d) for d in bom_state["bom_drivers"]]

# load Trend drivers
with open("../data/outputs/trend_state.json") as f:
    trend_state = json.load(f)
trend_drivers = [TechDriver(**d) for d in trend_state["trend_drivers"]]

print(f"BOM drivers: {len(bom_drivers)}")
print(f"Trend drivers: {len(trend_drivers)}")

BOM drivers: 27
Trend drivers: 29


In [2]:
# format driver lists for the merge prompt — use indices instead of IDs to prevent hallucination
bom_list = "\n".join([f"- Index: {i} | Name: {d.name} | Desc: {d.description[:150]}" for i, d in enumerate(bom_drivers)])
trend_list = "\n".join([f"- Index: {i} | Name: {d.name} | Desc: {d.description[:150]}" for i, d in enumerate(trend_drivers)])

merge_prompt = prompts.MERGE_DRIVERS.format(bom_drivers=bom_list, trend_drivers=trend_list)
# replace "id" references with "index" in the prompt for clarity
merge_prompt = merge_prompt.replace("bom_driver_id", "bom_driver_index").replace("trend_driver_id", "trend_driver_index")
merge_prompt = merge_prompt.replace("bom driver ids", "bom driver indices").replace("trend driver ids", "trend driver indices")

merge_result = safe_chat_json(merge_prompt, system="You are merging two technology driver lists for regulatory frequency monitoring. Use the INDEX numbers provided, not names or made-up IDs.")

# validate returned indices
matched_bom_indices = set()
matched_trend_indices = set()

print("=== MATCHES (both sources → validated) ===")
valid_matches = []
for m in merge_result.get("matches", []):
    bi = m.get("bom_driver_index")
    ti = m.get("trend_driver_index")
    if bi is None or not isinstance(bi, int) or bi < 0 or bi >= len(bom_drivers):
        print(f"  ⚠ BOM index '{bi}' invalid — skipping match")
        continue
    if ti is None or not isinstance(ti, int) or ti < 0 or ti >= len(trend_drivers):
        print(f"  ⚠ Trend index '{ti}' invalid — skipping match")
        continue
    valid_matches.append(m)
    matched_bom_indices.add(bi)
    matched_trend_indices.add(ti)
    print(f"  ✓ '{m.get('unified_name', '?')}'")
    print(f"    BOM[{bi}]: {bom_drivers[bi].name[:40]} ↔ Trend[{ti}]: {trend_drivers[ti].name[:40]}")
    print()
merge_result["matches"] = valid_matches

# any indices that weren't matched and weren't listed as solo
returned_bom_only = set()
for idx in merge_result.get("bom_only", []):
    if isinstance(idx, int) and 0 <= idx < len(bom_drivers):
        returned_bom_only.add(idx)
returned_trend_only = set()
for idx in merge_result.get("trend_only", []):
    if isinstance(idx, int) and 0 <= idx < len(trend_drivers):
        returned_trend_only.add(idx)

# add any drivers the LLM forgot to categorize
forgotten_bom = set(range(len(bom_drivers))) - matched_bom_indices - returned_bom_only
forgotten_trend = set(range(len(trend_drivers))) - matched_trend_indices - returned_trend_only
if forgotten_bom:
    print(f"⚠ {len(forgotten_bom)} BOM drivers not categorized by LLM — adding as BOM-only")
    returned_bom_only |= forgotten_bom
if forgotten_trend:
    print(f"⚠ {len(forgotten_trend)} Trend drivers not categorized by LLM — adding as Trend-only")
    returned_trend_only |= forgotten_trend

merge_result["bom_only"] = list(returned_bom_only)
merge_result["trend_only"] = list(returned_trend_only)

print(f"\n=== BOM-ONLY ({len(merge_result['bom_only'])} drivers) ===")
for bi in sorted(merge_result["bom_only"]):
    print(f"  • [{bi}] {bom_drivers[bi].name}")

print(f"\n=== TREND-ONLY ({len(merge_result['trend_only'])} drivers) ===")
for ti in sorted(merge_result["trend_only"]):
    print(f"  • [{ti}] {trend_drivers[ti].name}")

=== MATCHES (both sources → validated) ===
  ✓ 'Advanced Geolocation Techniques Using Distributed Sensors (TDOA, AOA)'
    BOM[25]: TDOA Radiolocation Module ↔ Trend[25]: Advanced Geolocation Techniques Using Di

  ✓ 'Wideband Direction Finding and Advanced Geolocation Techniques'
    BOM[19]: Wideband Direction Finding Capability ↔ Trend[25]: Advanced Geolocation Techniques Using Di

  ✓ 'High-Bandwidth I/Q Data Streaming Interfaces and Edge Computing'
    BOM[18]: I/Q Data Streaming Interfaces ↔ Trend[7]: Edge Computing for Spectrum Sensing

  ✓ 'Real-Time FPGA/ASIC Pipeline and Edge Computing for Spectrum Sensing'
    BOM[3]: Real-Time FPGA and ASIC Pipeline ↔ Trend[7]: Edge Computing for Spectrum Sensing

  ✓ 'FFT Processing Engine and Cognitive Radio Spectrum Sensing Technologies'
    BOM[10]: Fast Fourier Transform (FFT) Processing  ↔ Trend[13]: Cognitive Radio and Spectrum Sensing Tec

  ✓ 'Real-Time Spectrum Path and Cognitive Radio Spectrum Sensing'
    BOM[13]: Real-time spec

In [3]:
# build unified driver list with confidence tags
unified_drivers: list[TechDriver] = []

# matched drivers → high confidence
for m in merge_result.get("matches", []):
    bi = m["bom_driver_index"]
    ti = m["trend_driver_index"]
    bom_d = bom_drivers[bi]
    trend_d = trend_drivers[ti]
    merged_sources = list(set(bom_d.source_chunk_ids + trend_d.source_chunk_ids))
    driver = TechDriver(
        id=stable_id(m["unified_name"], "both"),
        name=m["unified_name"],
        description=f"BOM: {bom_d.description[:200]} | Trend: {trend_d.description[:200]}",
        origin=DriverOrigin.BOTH,
        confidence=DriverConfidence.HIGH,
        bom_node_id=bom_d.bom_node_id,
        source_chunk_ids=merged_sources,
        merge_reasoning=m.get("reasoning", ""),
    )
    unified_drivers.append(driver)

# BOM-only → medium confidence
for bi in merge_result.get("bom_only", []):
    d = bom_drivers[bi]
    d.confidence = DriverConfidence.MEDIUM
    unified_drivers.append(d)

# Trend-only → low confidence (speculative)
for ti in merge_result.get("trend_only", []):
    d = trend_drivers[ti]
    d.confidence = DriverConfidence.LOW
    unified_drivers.append(d)

print(f"=== Unified Driver List (before consolidation): {len(unified_drivers)} drivers ===")

=== Unified Driver List (before consolidation): 50 drivers ===


## Three-Stage Driver Consolidation

1. **Normalized name match** — catches "(DSA)" vs no-parentheses variants
2. **Cosine similarity at 0.70** — catches paraphrased duplicates
3. **LLM batch-grouping** — catches remaining semantic duplicates that cosine misses

In [4]:
import re
from collections import Counter

def normalize_name(name):
    name = name.lower().strip()
    name = re.sub(r'\s*\([^)]*\)\s*', ' ', name)
    return re.sub(r'\s+', ' ', name).strip()

# Stage 1: Normalized-name-match dedup
before_count = len(unified_drivers)
exact_seen: dict[str, int] = {}
for i, d in enumerate(unified_drivers):
    key = normalize_name(d.name)
    if key in exact_seen:
        first = unified_drivers[exact_seen[key]]
        first.source_chunk_ids = list(set(first.source_chunk_ids + d.source_chunk_ids))
        if d.confidence.value == "high" or (d.origin.value == "both" and first.origin.value != "both"):
            first.confidence = d.confidence
            first.origin = d.origin
        unified_drivers[i] = None
        print(f"  Stage 1: '{d.name}' merged into existing")
    else:
        exact_seen[key] = i
unified_drivers = [d for d in unified_drivers if d is not None]

dupes_removed = before_count - len(unified_drivers)
print(f"\nStage 1 — Normalized name dedup: {before_count} → {len(unified_drivers)} ({dupes_removed} removed)")


Stage 1 — Normalized name dedup: 50 → 50 (0 removed)


In [5]:
import numpy as np
from src.llm import embed

# Stage 2: Cosine similarity dedup at 0.85
SIMILARITY_THRESHOLD = 0.85

texts = [f"{d.name}: {d.description[:150]}" for d in unified_drivers]
embeddings = np.array(embed(texts))

norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
sim_matrix = (embeddings @ embeddings.T) / (norms @ norms.T + 1e-10)

merged_indices = set()
consolidated: list[TechDriver] = []

conf_rank = {"high": 3, "medium": 2, "low": 1}

for i in range(len(unified_drivers)):
    if i in merged_indices:
        continue
    cluster = [i]
    for j in range(i + 1, len(unified_drivers)):
        if j in merged_indices:
            continue
        if sim_matrix[i][j] > SIMILARITY_THRESHOLD:
            cluster.append(j)
            merged_indices.add(j)

    best_idx = max(cluster, key=lambda idx: (
        conf_rank.get(unified_drivers[idx].confidence.value, 0),
        len(unified_drivers[idx].source_chunk_ids),
    ))
    rep = unified_drivers[best_idx]

    all_sources = set()
    for idx in cluster:
        all_sources.update(unified_drivers[idx].source_chunk_ids)
    rep.source_chunk_ids = list(all_sources)

    consolidated.append(rep)
    if len(cluster) > 1:
        names = [unified_drivers[idx].name[:40] for idx in cluster]
        print(f"  Stage 2: Merged {len(cluster)}: {names} → {rep.name[:50]}")

print(f"\nStage 2 — Cosine dedup (>{SIMILARITY_THRESHOLD}): {len(unified_drivers)} → {len(consolidated)}")
unified_drivers = consolidated

# Stage 3: LLM batch-grouping for remaining semantic duplicates
driver_list = "\n".join([f"  {i}. {d.name}" for i, d in enumerate(unified_drivers)])

grouping_prompt = f"""You have {len(unified_drivers)} technology drivers for regulatory frequency monitoring.
Identify groups of drivers that describe the SAME or highly overlapping technology concept.

RULES:
- Only group drivers that are truly semantically redundant (same core technology, just named differently)
- Do NOT group drivers that are related but distinct (e.g., "Edge Computing" and "AI at the Edge" are similar but could be distinct)
- Do NOT merge BOM hardware components that are physically different subsystems
- If a driver is unique, it should not appear in any group

DRIVERS:
{driver_list}

Return JSON:
{{
  "groups": [
    {{
      "indices": [list of driver indices that should be merged],
      "best_name": "the best name for the merged driver",
      "reasoning": "why these are the same technology"
    }}
  ]
}}

Only include groups with 2+ members. If no duplicates remain, return {{"groups": []}}."""

group_result = safe_chat_json(grouping_prompt, system="You are deduplicating a technology driver list. Be conservative — only merge true duplicates.")

groups = group_result.get("groups", [])
print(f"\nStage 3 — LLM grouping found {len(groups)} duplicate groups:")

if groups:
    remove_indices = set()
    for g in groups:
        indices = g["indices"]
        best_name = g["best_name"]
        print(f"  Group: {[unified_drivers[i].name[:40] for i in indices]} → '{best_name}'")
        print(f"    Reason: {g.get('reasoning', '')[:100]}")

        # keep the one with highest confidence + most sources
        best_idx = max(indices, key=lambda idx: (
            conf_rank.get(unified_drivers[idx].confidence.value, 0),
            len(unified_drivers[idx].source_chunk_ids),
        ))
        rep = unified_drivers[best_idx]
        rep.name = best_name

        # merge source chunk IDs from all in group
        for idx in indices:
            if idx != best_idx:
                rep.source_chunk_ids = list(set(rep.source_chunk_ids + unified_drivers[idx].source_chunk_ids))
                remove_indices.add(idx)

    unified_drivers = [d for i, d in enumerate(unified_drivers) if i not in remove_indices]

print(f"\nStage 3 — LLM grouping: → {len(unified_drivers)} drivers")
print(f"\n{'='*60}")
print(f"TOTAL DEDUP: {before_count} → {len(unified_drivers)} drivers")
print(f"{'='*60}")

  Stage 2: Merged 2: ['Digital Downconverter (DDC)', 'Digital Downconverters (DDC)'] → Digital Downconverters (DDC)
  Stage 2: Merged 2: ['Bandwidth Extension Hardware Module R&S®', 'Bandwidth Extension Firmware Option R&S®'] → Bandwidth Extension Hardware Module R&S®HS1-BW500

Stage 2 — Cosine dedup (>0.85): 50 → 48



Stage 3 — LLM grouping found 4 duplicate groups:
  Group: ['Advanced Geolocation Techniques Using Di', 'Wideband Direction Finding and Advanced ', 'Measurement Data Processing and Triangul'] → 'Advanced Geolocation Techniques Using Distributed Sensors (TDOA, AOA)'
    Reason: Drivers 0, 1, and 9 all describe advanced geolocation techniques using distributed sensors and trian
  Group: ['Resampler / Digital Down Converter (DDC)', 'Parallel Digital Downconverters', 'Digital Downconverters (DDC)'] → 'Digital Downconverters (DDC) and Parallel Digital Downconverters'
    Reason: Drivers 11, 13, and 14 all refer to digital downconverter technology, including parallel implementat
  Group: ['AI-Driven Autonomous Spectrum Monitoring', 'Radio Frequency Machine Learning (RFML) ', 'AI and RF Machine Learning (RFML) for Au', 'Edge and Local Processing with RF Machin'] → 'AI and RF Machine Learning (RFML) for Autonomous Spectrum Monitoring'
    Reason: Drivers 25, 26, 30, and 44 all describe the use

In [6]:
# save consolidated unified drivers
print(f"=== Final Unified Driver List: {len(unified_drivers)} drivers ===\n")
for d in unified_drivers:
    origin_tag = {"both": "VALIDATED", "bom": "INCREMENTAL", "trend": "SPECULATIVE"}[d.origin.value]
    print(f"  [{d.confidence.value:6s}] [{origin_tag:11s}] {d.name}")

merge_state = {
    "unified_drivers": [d.model_dump(mode="json") for d in unified_drivers],
    "merge_result": merge_result,
}
with open("../data/outputs/merge_state.json", "w") as f:
    json.dump(merge_state, f, indent=2)
print(f"\nSaved {len(unified_drivers)} drivers to merge_state.json")

=== Final Unified Driver List: 40 drivers ===

  [high  ] [VALIDATED  ] Advanced Geolocation Techniques Using Distributed Sensors (TDOA, AOA)
  [high  ] [VALIDATED  ] High-Bandwidth I/Q Data Streaming Interfaces and Edge Computing
  [high  ] [VALIDATED  ] Real-Time FPGA/ASIC Pipeline and Edge Computing for Spectrum Sensing
  [high  ] [VALIDATED  ] FFT Processing Engine and Cognitive Radio Spectrum Sensing Technologies
  [high  ] [VALIDATED  ] Real-Time Spectrum Path and Cognitive Radio Spectrum Sensing
  [high  ] [VALIDATED  ] Wideband Analog-to-Digital Conversion and Wideband Spectrum Sensing with ML
  [high  ] [VALIDATED  ] Panorama Scan Functionality and Terahertz Frequency Band Utilization
  [high  ] [VALIDATED  ] Data Link Infrastructure and Integration of Big Data and Distributed Monitoring Networks
  [medium] [INCREMENTAL] Analog-to-Digital Converter (ADC)
  [medium] [INCREMENTAL] Detector
  [medium] [INCREMENTAL] Digital Downconverters (DDC) and Parallel Digital Downconverters
